# Does HFTD Eligibility Predict SGIP Battery Funding?

**Question:** Among residential battery storage applicants to California's Self-Generation
Incentive Program (SGIP) in PG&E territory, are applicants in a High Fire-Threat District
(HFTD Tier 2/3), who qualify for enhanced incentives, more likely to reach funded (`Paid`)
status than non-HFTD applicants in the same zip code who applied at the same time?

**Data:**
- SGIP Weekly Statewide Report: application-level data, downloaded from the SGIP website
- U.S. Census Bureau ACS 5-year estimates (2020–2024): housing units and median household
  income by ZIP Code Tabulation Area (ZCTA), from the Census Data API

See `plan.md` for the full analysis plan.

## 1. Setup

Imports and constants. The Census API key is read from a `.env` file (excluded from the repo
by `.gitignore`). To rerun this notebook, create a `.env` file next to it containing:

```
CENSUS_API_KEY=your_key_here
```

A free key can be requested at https://api.census.gov/data/key_signup.html.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# Load variables from .env into the environment, then read the Census key
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
if not CENSUS_API_KEY:
    raise RuntimeError("CENSUS_API_KEY not found. Add it to a .env file (see above).")

# Folder for downloaded raw files (gitignored)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# SGIP report URL: always serves the latest weekly snapshot as an .xlsx file
SGIP_REPORT_URL = "https://www.selfgenca.com/documents/reports/statewide_projects"

# ACS 5-year vintage (2024 = 2020-2024 estimates) and the variables to pull,
# mapped from Census variable codes to readable column names
ACS_YEAR = 2024
ACS_VARIABLES = {
    "B25001_001E": "housing_units",            # Total housing units
    "B19013_001E": "median_household_income",  # Median household income (2024 dollars)
}

## 2. Pull the SGIP application data

SGIP does not offer a formal API, but the report URL returns the latest weekly `.xlsx`
directly, so the notebook downloads it with `requests` and no manual download is needed.

The report is replaced every week and application statuses keep changing, so the file is
saved under its original dated filename and the snapshot date is printed. Results are tied to
that snapshot. If a file with the same name is already in `data/`, the download is skipped.

In [2]:
def download_sgip_report(url, data_dir):
    """Download the current SGIP weekly report and return its local path.

    Parameters
    ----------
    url : str
        SGIP report URL that serves the latest .xlsx snapshot.
    data_dir : Path
        Folder where the file is saved.
    """
    # stream=True: read the headers (to get the dated filename) before downloading the body;
    # timeout=120: seconds to wait for the server before giving up
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        # Header looks like: attachment; filename="Weekly Statewide Report_09_27_2026(posted 09_27_26).xlsx"
        disposition = response.headers["Content-Disposition"]
        filename = re.search(r'filename="(.+?)"', disposition).group(1)
        local_path = data_dir / filename

        if local_path.exists():
            print(f"Already downloaded: {filename}")
            return local_path

        with open(local_path, "wb") as file:
            # chunk_size=1 MB: write the ~25 MB file in pieces rather than holding it in memory
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)
    print(f"Downloaded: {filename}")
    return local_path


sgip_path = download_sgip_report(SGIP_REPORT_URL, DATA_DIR)

# Snapshot date is the first MM_DD_YYYY in the filename
snapshot_date = pd.to_datetime(
    re.search(r"\d{2}_\d{2}_\d{4}", sgip_path.name).group(), format="%m_%d_%Y"
)
print(f"SGIP snapshot date: {snapshot_date:%B %d, %Y}")

Already downloaded: Weekly Statewide Report_09_27_2026(posted 09_27_26).xlsx
SGIP snapshot date: September 27, 2026


The data is on the first sheet. Rows 1–3 are a title, a date stamp and a blank row, so the
column headers are on Excel row 4 (`header=3`, counting from zero).

In [3]:
# sheet_name=0: first sheet (the data; the second sheet is a field dictionary)
# header=3: column names are on Excel row 4
sgip_raw = pd.read_excel(sgip_path, sheet_name=0, header=3)

print(f"{len(sgip_raw):,} rows, {sgip_raw.shape[1]} columns")
sgip_raw.head()

106,585 rows, 51 columns


,SGIP Administrator,Application Code,Program Year,Step,Budget Category,Fully Qualified State,Equipment Type,Rated Capacity [kW],Energy Storage Capacity (kWh),Fuel Type,...,Cancelled Date,CSI Rating,Advance Payment Program (AB 209),Total Eligible Project Cost - Solar,Total Eligible Project Cost - Storage,Solar Array Manufacturers,Solar Inverter Manufacturers,Funding Source,"Located in HFTD (Tier 2, Tier 3)",Experienced Two PSPS Events (True/False)
0,Southern California Edison,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,750.0,1500.0,NaN,...,05/04/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
1,Southern California Edison,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,125.0,250.0,NaN,...,05/04/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
2,Center for Sustainable Energy,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,261.0,522.0,NaN,...,06/01/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
3,LA Dept. of Water and Power,LA-SGIP-2025-0001,2025.0,6.0,Residential Solar and Storage Equity - AB 209,RRF Technical Review,Electrochemical Storage,23.0,26.4,NaN,...,NaN,11.050649,Yes,48938.59,41485.71,Mission Solar Energy LLC,Tesla Inc.,State,Tier 2,Not Applicable
4,LA Dept. of Water and Power,LA-SGIP-2025-0002,2025.0,6.0,Residential Solar and Storage Equity - AB 209,RRF Suspended,Electrochemical Storage,34.5,39.6,NaN,...,NaN,23.566935,Yes,104367.86,47142.86,Tesla Inc.,Tesla Inc.,State,Not Applicable,Not Applicable


## 3. Pull Census ACS data by ZCTA

The Census Data API returns JSON: a list of rows whose first row is the column headers.
Since the 2020 ACS, ZCTAs cannot be filtered by state in the request, so all ZCTAs
nationwide are pulled here. The data is narrowed to California in the cleaning step.

In [4]:
def fetch_acs_zcta(variables, year, api_key):
    """Pull ACS 5-year estimates for every ZCTA and return them as a DataFrame.

    Parameters
    ----------
    variables : dict
        Census variable codes mapped to the column names to use.
    year : int
        ACS 5-year vintage (e.g. 2024 = 2020-2024 estimates).
    api_key : str
        Census Data API key.
    """
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": ",".join(variables),                # variables to return
        "for": "zip code tabulation area:*",       # geography: all ZCTAs
        "key": api_key,
    }
    # timeout=120: seconds to wait for the server before giving up
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    header, *rows = response.json()
    acs = pd.DataFrame(rows, columns=header)
    return acs.rename(columns={**variables, "zip code tabulation area": "zcta"})


acs_raw = fetch_acs_zcta(ACS_VARIABLES, ACS_YEAR, CENSUS_API_KEY)

print(f"{len(acs_raw):,} ZCTAs")
acs_raw.head()

33,772 ZCTAs


,housing_units,median_household_income,zcta
0,7570,19454,00601
1,17610,21420,00602
2,26239,20933,00603
3,2681,20992,00606
4,12636,24496,00610


## 4. Clean the SGIP data

The sample is narrowed one filter at a time, and the number of rows left after each step is
recorded so every drop can be traced:

1. **PG&E only**: the research question covers PG&E territory, which also contains Ava's
   service area.
2. **Residential sectors**: `Residential`, `Single Family`, `Multifamily`.
3. **Storage projects**: `Energy Storage Capacity (kWh)` is recorded.
4. **Drop `RRF Rejected`**: these applications were rejected at intake and never assigned a
   budget, so they never had a chance to be funded.
5. **Program Year 2024 or later**: CPUC Decision 24-03-071 / AB 209 restructured the
   residential equity budgets in 2024. Keeping only the new rules avoids mixing two
   eligibility regimes.
6. **HFTD status recorded**: drop rows where `Located in HFTD` is blank.
7. **Received before 2026**: applications received in 2026 are almost all still in review
   (Paid rate near 0%). Keeping them would make whichever group applied later look less
   successful just because of timing.

`Date Received` is stored as text like `01/02/24 10:59:54.708805 PST` (or `PDT`), so the
timezone label is stripped before parsing.

In [5]:
# Long SGIP column names used several times below
HFTD_COL = "Located in HFTD (Tier 2, Tier 3)"
PSPS_COL = "Experienced Two PSPS Events (True/False)"

RESIDENTIAL_SECTORS = ["Residential", "Single Family", "Multifamily"]
HFTD_STATUSES = ["Tier 2", "Tier 3", "Not Applicable"]
FIRST_PROGRAM_YEAR = 2024                       # first year under D.24-03-071 / AB 209 rules
RECEIVED_CUTOFF = pd.Timestamp("2026-01-01")    # later applications are mostly unresolved


def parse_received_date(date_text):
    """Convert SGIP 'Date Received' strings to timestamps.

    Parameters
    ----------
    date_text : pd.Series
        Raw strings such as '01/02/24 10:59:54.708805 PST'.
    """
    # Remove the trailing PST/PDT label, which pandas cannot parse with a fixed format
    without_zone = date_text.str.replace(r"\s+P[SD]T$", "", regex=True)
    # errors="coerce": unparseable strings become NaT so they can be counted, not crash
    return pd.to_datetime(without_zone, format="%m/%d/%y %H:%M:%S.%f", errors="coerce")


def apply_filters(df, filters):
    """Apply filters in order and record the rows left after each one.

    Parameters
    ----------
    df : pd.DataFrame
        Data to filter.
    filters : list of (str, callable)
        Each item is a step description and a function that takes the current
        DataFrame and returns a True/False mask of rows to keep.

    Returns the filtered DataFrame and a table of row counts by step.
    """
    log = [("Raw SGIP report", len(df))]
    for description, keep in filters:
        df = df[keep(df)]
        log.append((description, len(df)))

    funnel = pd.DataFrame(log, columns=["step", "rows_remaining"])
    # Rows removed by each step (zero for the starting row)
    funnel["rows_dropped"] = (-funnel["rows_remaining"].diff()).fillna(0).astype(int)
    return df, funnel

In [6]:
sgip = sgip_raw.assign(received=parse_received_date(sgip_raw["Date Received"]))

sample_filters = [
    ("1. PG&E administrator",
     lambda df: df["SGIP Administrator"] == "Pacific Gas and Electric"),
    ("2. Residential sector",
     lambda df: df["Host Customer Sector"].isin(RESIDENTIAL_SECTORS)),
    ("3. Storage project (kWh recorded)",
     lambda df: df["Energy Storage Capacity (kWh)"].notna()),
    ("4. Drop RRF Rejected",
     lambda df: df["Fully Qualified State"] != "RRF Rejected"),
    (f"5. Program Year >= {FIRST_PROGRAM_YEAR}",
     lambda df: df["Program Year"] >= FIRST_PROGRAM_YEAR),
    ("6. HFTD status recorded",
     lambda df: df[HFTD_COL].isin(HFTD_STATUSES)),
    (f"7. Received before {RECEIVED_CUTOFF:%Y-%m-%d}",
     lambda df: df["received"] < RECEIVED_CUTOFF),
]

sgip_sample, filter_funnel = apply_filters(sgip, sample_filters)
filter_funnel

,step,rows_remaining,rows_dropped
0,Raw SGIP report,106585,0
1,1. PG&E administrator,47990,58595
2,2. Residential sector,43526,4464
3,3. Storage project (kWh recorded),43460,66
4,4. Drop RRF Rejected,41587,1873
5,5. Program Year >= 2024,9970,31617
6,6. HFTD status recorded,9447,523
7,7. Received before 2026-01-01,9110,337


### Analysis variables

Each application gets the variables used in the charts and regressions:

| Variable | Definition |
|---|---|
| `paid` | 1 if `Budget Classification` is `Paid` (the outcome), else 0 |
| `hftd` | 1 if HFTD Tier 2 or Tier 3, 0 if Not Applicable (the treatment) |
| `tier2`, `tier3` | Separate tier indicators, for the robustness check |
| `psps` | 1 if the household experienced two PSPS (Public Safety Power Shutoff) events, else 0. This is a second route to enhanced incentives, so it is controlled for. |
| `received_quarter` | Calendar quarter the application was received (timing control) |
| `zip5` | 5-digit zip code. A few zips are recorded as ZIP+4, so only the first 5 digits are kept. |

In [7]:
applications = pd.DataFrame({
    "application_code": sgip_sample["Application Code"],
    "zip5": sgip_sample["Zip"].astype(str).str[:5],
    "received": sgip_sample["received"],
    "received_quarter": sgip_sample["received"].dt.to_period("Q").astype(str),
    "hftd_status": sgip_sample[HFTD_COL],
    "hftd": sgip_sample[HFTD_COL].isin(["Tier 2", "Tier 3"]).astype(int),
    "tier2": sgip_sample[HFTD_COL].eq("Tier 2").astype(int),
    "tier3": sgip_sample[HFTD_COL].eq("Tier 3").astype(int),
    "psps": sgip_sample[PSPS_COL].eq("Yes").astype(int),
    "budget_classification": sgip_sample["Budget Classification"],
    "paid": sgip_sample["Budget Classification"].eq("Paid").astype(int),
}).reset_index(drop=True)

# Sanity checks: one row per application, and no missing key fields
assert applications["application_code"].is_unique
assert applications[["zip5", "received", "budget_classification"]].notna().all().all()

print(f"{len(applications):,} applications in {applications['zip5'].nunique():,} zip codes")
applications.groupby("hftd_status").agg(
    applications=("paid", "size"),
    paid_rate=("paid", "mean"),
    psps_share=("psps", "mean"),
).round(3)

9,110 applications in 711 zip codes


,applications,paid_rate,psps_share
hftd_status,,,
Not Applicable,6589,0.353,0.177
Tier 2,1751,0.509,0.572
Tier 3,770,0.453,0.694


Zip code fixed effects compare HFTD and non-HFTD applicants **within the same zip**, so only
zips that contain both kinds of applicants provide the identifying variation.

In [8]:
# For each zip: lowest and highest value of hftd (0/0 = none, 1/1 = all, 0/1 = both)
zip_hftd_range = applications.groupby("zip5")["hftd"].agg(["min", "max"])
zips_with_both = ((zip_hftd_range["min"] == 0) & (zip_hftd_range["max"] == 1)).sum()

print(f"Zips with any HFTD applicant:     {(zip_hftd_range['max'] == 1).sum():,}")
print(f"Zips with any non-HFTD applicant: {(zip_hftd_range['min'] == 0).sum():,}")
print(f"Zips with both (identifying):     {zips_with_both:,}")

Zips with any HFTD applicant:     351
Zips with any non-HFTD applicant: 594
Zips with both (identifying):     234


## 5. Clean the Census data and merge

- Keep California ZCTAs (codes `90000`–`96199`).
- Convert estimates from text to numbers. The Census API reports estimates it cannot compute
  as large negative codes (e.g. `-666666666` for median income in very small ZCTAs). These
  are set to missing.
- Merge onto the applications by 5-digit zip. Zip codes and ZCTAs are not identical: some
  zips, such as PO-box-only zips, have no ZCTA. Unmatched applications are counted below.

In [9]:
CA_ZCTA_RANGE = ("90000", "96199")   # first and last California ZCTA codes

acs = acs_raw[acs_raw["zcta"].between(*CA_ZCTA_RANGE)].copy()
for column in ACS_VARIABLES.values():
    acs[column] = pd.to_numeric(acs[column])
    # mask(...): replace negative "cannot compute" codes with NaN
    acs[column] = acs[column].mask(acs[column] < 0)

print(f"{len(acs):,} California ZCTAs; "
      f"{acs['median_household_income'].isna().sum()} with no median income estimate")

# how="left": keep every application, even if its zip has no ZCTA match;
# validate="many_to_one": each ZCTA must appear only once in the Census data;
# indicator=True: adds a _merge column showing whether each row matched
applications = applications.merge(
    acs, how="left", left_on="zip5", right_on="zcta",
    validate="many_to_one", indicator=True,
)
unmatched = applications["_merge"] == "left_only"
print(f"Applications with no matching ZCTA: {unmatched.sum():,} "
      f"({applications.loc[unmatched, 'zip5'].nunique()} zips)")
applications = applications.drop(columns=["zcta", "_merge"])

1,802 California ZCTAs; 212 with no median income estimate
Applications with no matching ZCTA: 14 (11 zips)


### Zip-level summary (secondary question)

The applications are collapsed to one row per zip: the number of storage applications, the
share of them in an HFTD, and applications per 1,000 housing units. The rate is left missing
for zips with no ZCTA match or zero housing units.

In [10]:
zip_summary = (
    applications.groupby("zip5")
    .agg(
        applications=("paid", "size"),
        share_hftd=("hftd", "mean"),
        housing_units=("housing_units", "first"),
        median_household_income=("median_household_income", "first"),
    )
    .reset_index()
)
# where(... > 0): treat zero housing units as missing so the rate is not divided by zero
zip_summary["applications_per_1000_units"] = (
    1000 * zip_summary["applications"]
    / zip_summary["housing_units"].where(zip_summary["housing_units"] > 0)
)

zip_summary.describe().round(2)

,applications,share_hftd,housing_units,median_household_income,applications_per_1000_units
count,711.00,711.00,700.00,679.00,697.00
mean,12.81,0.31,8550.77,108753.08,2.90
std,14.44,0.40,7289.44,48929.40,4.83
min,1.00,0.00,0.00,13333.00,0.04
25%,3.00,0.00,1851.00,72467.50,0.81
50%,8.00,0.00,6903.00,98125.00,1.60
75%,18.00,0.67,13823.00,131321.50,3.11
max,91.00,1.00,31396.00,250001.00,62.50
